In [1]:
import pandas as pd
import folium
from folium import plugins
import numpy as np
from scipy.spatial.distance import cdist

import calliope

In [2]:
coordinates_df=pd.read_csv('data_tables/nodes_base_info.csv')
demand_nodes=coordinates_df[coordinates_df['nodes'].str.contains('D')].copy()
heat_transmission_nodes=coordinates_df[coordinates_df['nodes'].str.contains('TH')].copy()
electricity_transmission_nodes=coordinates_df[coordinates_df['nodes'].str.contains('TE')].copy()

# Extract coordinates as arrays
demand_coords=demand_nodes[['latitude', 'longitude']].values
transmission_coords=heat_transmission_nodes[['latitude', 'longitude']].values

# Calculate distances from each demand node to all heat transmission nodes
heat_distances=cdist(demand_coords, transmission_coords, metric='euclidean')

# Find index of nearest heat transmission node for each demand node
nearest_transmission_nodes_idx=np.argmin(heat_distances, axis=1)
demand_nodes['heat_node']=heat_transmission_nodes.iloc[nearest_transmission_nodes_idx]['nodes'].values
demand_nodes['electricity_node']=electricity_transmission_nodes.iloc[nearest_transmission_nodes_idx]['nodes'].values

# Create links dataframe and write to csv
heat_links = pd.DataFrame({
    'techs': demand_nodes['nodes'] + '_to_' + demand_nodes['heat_node'],
    'color': '#823739',
    'name': 'Heat distribution',
    'base_tech': 'transmission',
    'flow_cap_max': '2000',
    'flow_out_eff_per_distance': '0.98',
    'lifetime': '20',
    'link_to': demand_nodes['nodes'],
    'link_from': demand_nodes['heat_node']
}).reset_index(drop=True)

electricity_links=pd.DataFrame({
    'techs': demand_nodes['nodes'] + '_to_' + demand_nodes['electricity_node'],
    'color': '#6783E3',
    'name': 'Electricity distribution',
    'base_tech': 'transmission',
    'flow_cap_max': '2000',
    'flow_out_eff_per_distance': '0.99',
    'lifetime': '20',
    'link_to': demand_nodes['nodes'],
    'link_from': demand_nodes['electricity_node']
}).reset_index(drop=True)

distribution_techs=pd.concat([heat_links, electricity_links], ignore_index=True)

transmission_network=pd.read_csv('data_tables/transmission_network.csv')
updated_links = pd.concat([transmission_network, distribution_techs], ignore_index=True)
updated_links.to_csv('data_tables/links.csv', index=False)

# Create carrier dataframes and write to csv
distribution_heat=pd.DataFrame({
    'techs': demand_nodes['nodes'] + '_to_' +  demand_nodes['heat_node'],
    'carrier_out': '1',
    'carrier_in': '1'
    }).reset_index(drop=True)

transmission_heat=pd.read_csv('data_tables/transmission_heat.csv')
updated_heat_links=pd.concat([transmission_heat, distribution_heat], ignore_index=True)
updated_heat_links.to_csv('data_tables/links_heat.csv', index=False)

distribution_electricity=pd.DataFrame({
    'techs': demand_nodes['nodes'] + '_to_' +  demand_nodes['electricity_node'],
    'carrier_out': '1',
    'carrier_in': '1'
    }).reset_index(drop=True)

transmission_electricity=pd.read_csv('data_tables/transmission_electricity.csv')
updated_electricity_links=pd.concat([transmission_electricity, distribution_electricity], ignore_index=True)
updated_electricity_links.to_csv('data_tables/links_electricity.csv', index=False)

# Create costs dataframe and write to csv
distribution_heat_costs=pd.DataFrame({
    'techs': demand_nodes['nodes']  + '_to_' +  demand_nodes['heat_node'],
    'cost_flow_cap_per_distance': '100'
    }).reset_index(drop=True)

distribution_electricity_costs=pd.DataFrame({
    'techs': demand_nodes['nodes']  + '_to_' +  demand_nodes['electricity_node'],
    'cost_flow_cap_per_distance': '50'
    }).reset_index(drop=True)

distribution_costs=pd.concat([distribution_heat_costs, distribution_electricity_costs], ignore_index=True)

transmission_costs=pd.read_csv('data_tables/transmission_costs.csv')
updated_links_costs = pd.concat([transmission_costs, distribution_costs], ignore_index=True)
updated_links_costs.to_csv('data_tables/links_costs.csv', index=False)


In [4]:
calliope.set_log_verbosity("INFO", include_solver_output=True)

model = calliope.read_yaml("model.yaml")

[2025-11-11 21:41:37] INFO     Math init | loading pre-defined math.
[2025-11-11 21:41:37] INFO     Math init | loading math files {'base', 'spores', 'storage_inter_cluster', 'operate', 'milp'}.
[2025-11-11 21:41:37] INFO     Model: preprocessing data
[2025-11-11 21:41:37] INFO     Math build | building applied math with ['base'].
[2025-11-11 21:41:46] INFO     input data `color` not defined in model math; it will not be available in the optimisation problem.
[2025-11-11 21:41:46] INFO     input data `name` not defined in model math; it will not be available in the optimisation problem.
[2025-11-11 21:41:46] INFO     input data `comment` not defined in model math; it will not be available in the optimisation problem.
[2025-11-11 21:41:46] INFO     input data `link_to` not defined in model math; it will not be available in the optimisation problem.
[2025-11-11 21:41:46] INFO     input data `link_from` not defined in model math; it will not be available in the optimisation problem.
[2025

In [19]:
model.inputs

print(model.inputs.techs)

<xarray.DataArray 'techs' (techs: 455)> Size: 4kB
array(['D100_to_TE53', 'D100_to_TH53', 'D101_to_TE52', ..., 'demand_heat',
       'supply_electricity', 'supply_geothermal'], dtype=object)
Coordinates:
  * techs    (techs) object 4kB 'D100_to_TE53' ... 'supply_geothermal'


In [6]:
model.inputs.flow_cap_max.to_series().dropna()

techs
D100_to_TE53          2000.0
D100_to_TH53          2000.0
D101_to_TE52          2000.0
D101_to_TH52          2000.0
D102_to_TE52          2000.0
                       ...  
TH7_to_TH8            2000.0
TH8_to_TH9            2000.0
TH9_to_TH10           2000.0
supply_electricity    2000.0
supply_geothermal     2000.0
Name: flow_cap_max, Length: 453, dtype: float64

In [7]:
model.inputs.sink_use_equals.sum(
    "timesteps", min_count=1, skipna=True
).to_series().dropna()

nodes  techs      
D1     demand_heat    10.0
D10    demand_heat    10.0
D100   demand_heat    10.0
D101   demand_heat    10.0
D102   demand_heat    10.0
                      ... 
TH71   demand_heat     0.0
TH72   demand_heat     0.0
TH73   demand_heat     0.0
TH8    demand_heat     0.0
TH9    demand_heat     0.0
Name: sink_use_equals, Length: 291, dtype: float64

In [8]:
model.build()
model.solve()

[2025-11-11 21:42:06] INFO     Model: backend build starting
[2025-11-11 21:42:06] INFO     Optimisation Model | parameters/lookups | Generated.
[2025-11-11 21:42:06] INFO     Optimisation Model | variables | Generated.
[2025-11-11 21:42:08] INFO     Optimisation Model | global_expressions | Generated.
[2025-11-11 21:42:09] INFO     Optimisation Model | constraints | Generated.
[2025-11-11 21:42:09] INFO     Optimisation Model | piecewise_constraints | Generated.
[2025-11-11 21:42:10] INFO     Optimisation Model | objectives | Generated.
[2025-11-11 21:42:10] INFO     Model: backend build complete
[2025-11-11 21:42:10] INFO     Optimisation model | starting model in base mode.
[2025-11-11 21:42:10] DEBUG    Set parameter Username
Set parameter LicenseID to value 2716243
Academic license - for non-commercial use only - expires 2026-09-30
Read LP format model from file C:\Users\alexn\AppData\Local\Temp\tmpsta6p9ze.pyomo.lp
Reading time = 0.01 seconds
x1: 4204 rows, 4645 columns, 11114 no

In [9]:
model.results

<xarray.Dataset> Size: 21MB
Dimensions:                     (nodes: 296, techs: 455, carriers: 2,
                                 timesteps: 1, costs: 1)
Coordinates:
  * techs                       (techs) object 4kB 'D100_to_TE53' ... 'supply...
  * nodes                       (nodes) object 2kB 'D1' 'D10' ... 'TH8' 'TH9'
  * carriers                    (carriers) object 16B 'electricity' 'heat'
  * timesteps                   (timesteps) datetime64[ns] 8B 2050-01-01
  * costs                       (costs) object 8B 'monetary'
Data variables: (12/20)
    flow_cap                    (nodes, techs, carriers) float64 2MB nan ... nan
    link_flow_cap               (techs) float64 4kB 0.0 10.0 0.0 ... nan nan nan
    flow_out                    (nodes, techs, carriers, timesteps) float64 2MB ...
    flow_in                     (nodes, techs, carriers, timesteps) float64 2MB ...
    source_use                  (nodes, techs, timesteps) float64 1MB nan ......
    source_cap                  (nodes, techs) float64 1MB nan nan ... nan nan
    ...                          ...
    min_cost_optimisation       float64 8B 261.8
    capacity_factor             (nodes, techs, carriers, timesteps) float64 2MB ...
    systemwide_capacity_factor  (techs, carriers) float64 7kB 0.0 0.0 ... 1.0
    systemwide_levelised_cost   (techs, costs, carriers) float64 7kB nan ... ...
    total_levelised_cost        (costs, carriers) float64 16B 0.5229 0.1747
    unmet_sum                   (nodes, carriers, timesteps) float64 5kB 0.0 ...

In [10]:
df_heat = (
    model.results.flow_out.sel(carriers="heat")
    .sum("nodes", min_count=1, skipna=True)
    .to_series()
    .dropna()
    .unstack("techs")
)

df_heat.head()

techs,D100_to_TH53,D101_to_TH52,D102_to_TH52,D103_to_TH48,D104_to_TH48,D105_to_TH73,D106_to_TH72,D107_to_TH72,D108_to_TH71,D109_to_TH71,...,TH69_to_TH70,TH6_to_TH7,TH70_to_TH71,TH71_to_TH72,TH72_to_TH73,TH73_to_TH52,TH7_to_TH8,TH8_to_TH9,TH9_to_TH10,supply_geothermal
timesteps,,,,,,,,,,,,,,,,,,,,,
2050-01-01,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,...,30.022268,1458.258601,50.036121,80.05281,100.073329,130.10512,1457.408113,1454.671005,1449.283772,1498.653822


In [11]:
df_electricity = (
    model.results.flow_out.sel(carriers="electricity")
    .sum("nodes", min_count=1, skipna=True)
    .to_series()
    .dropna()
    .unstack("techs")
)

df_electricity.head()

techs,D100_to_TE53,D101_to_TE52,D102_to_TE52,D103_to_TE48,D104_to_TE48,D105_to_TE73,D106_to_TE72,D107_to_TE72,D108_to_TE71,D109_to_TE71,...,TE69_to_TE70,TE6_to_TE7,TE70_to_TE71,TE71_to_TE72,TE72_to_TE73,TE73_to_TE52,TE7_to_TE8,TE8_to_TE9,TE9_to_TE10,supply_electricity
timesteps,,,,,,,,,,,,,,,,,,,,,
2050-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,500.645306


In [12]:
costs = model.results.cost.to_series().dropna()
costs.head()

nodes  techs         costs   
D1     D1_to_TE30    monetary    0.000000
       D1_to_TH30    monetary    0.003070
D10    D10_to_TE19   monetary    0.000000
       D10_to_TH19   monetary    0.000473
D100   D100_to_TE53  monetary    0.000000
Name: cost, dtype: float64

In [13]:
# We set the color mapping to use in all our plots by extracting the colors defined in the technology definitions of our model.
colors = model.inputs.color.to_series().to_dict()

df_electricity1 = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers="electricity")
    .sum("nodes")
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_electricity_demand = df_electricity1[df_electricity1.techs == "demand_electricity"]
df_electricity_other = df_electricity1[df_electricity1.techs != "demand_electricity"]

print(df_electricity1.head())


                techs  timesteps  Flow in/out (kWh)
0          SE1_to_TE1 2050-01-01          -0.118118
1          SH1_to_TE1 2050-01-01          -0.527188
2  demand_electricity 2050-01-01        -500.000000
3  supply_electricity 2050-01-01         500.645306


In [14]:
carriers = ["heat", "electricity"]
df_flows = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers=carriers)
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_demand = df_flows[df_flows.techs.str.contains("demand")]
df_flows_other = df_flows[~df_flows.techs.str.contains("demand")]

print(df_flows.head())

  nodes         techs carriers  timesteps  Flow in/out (kWh)
0    D1    D1_to_TH30     heat 2050-01-01               10.0
1    D1   demand_heat     heat 2050-01-01              -10.0
2   D10   D10_to_TH19     heat 2050-01-01               10.0
3   D10   demand_heat     heat 2050-01-01              -10.0
4  D100  D100_to_TH53     heat 2050-01-01               10.0


In [15]:
df_capacity = (
    model.results.flow_cap.where(
        ~model.inputs.base_tech.str.contains("demand|transmission")
    )
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)

In [16]:
df_coords = model.inputs[["latitude", "longitude"]].to_dataframe().reset_index()
df_capacity = (
    model.results.flow_cap.where(model.inputs.base_tech == "transmission")
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)

# Define distribution and transmission dataframes for plotting
df_capacity_coords = pd.merge(df_coords, df_capacity, left_on="nodes", right_on="nodes").sort_values(by=['techs'])

# Extract link information from techs column (format: "node_from_to_node_to")
df_links = df_capacity_coords.copy()

# Split the techs column to get link_from and link_to
df_links[['link_from', 'link_to']] = df_links['techs'].str.rsplit('_to_', n=1, expand=True)

# Merge with df_coords twice to get both from and to coordinates
# First merge for "from" coordinates
df_links = df_links.merge(
    df_coords[['nodes', 'latitude', 'longitude']],
    left_on='link_from',
    right_on='nodes',
    how='left',
    suffixes=('', '_from')
)
df_links = df_links.rename(columns={'latitude': 'lat_from', 'longitude': 'lon_from'})

# Second merge for "to" coordinates
df_links = df_links.merge(
    df_coords[['nodes', 'latitude', 'longitude']],
    left_on='link_to',
    right_on='nodes',
    how='left',
    suffixes=('_temp', '_to')
)
df_links = df_links.rename(columns={'latitude': 'lat_to', 'longitude': 'lon_to'})

# Clean up duplicate columns
df_links = df_links.drop(columns=['nodes_temp', 'nodes_to'], errors='ignore')

# Create a Folium map centered on your data
center_lat = df_coords['latitude'].mean()
center_lon = df_coords['longitude'].mean()

map_fig = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=16,
    tiles='OpenStreetMap'
)


# Add lines for each link
for idx, row in df_links.iterrows():
    folium.PolyLine(
        locations=[[row['lat_from'], row['lon_from']], [row['lat_to'], row['lon_to']]],
        color='blue',
        weight=1,
        opacity=0.7,
        popup=f"<b>{row['techs']}</b><br>From: {row['link_from']}<br>To: {row['link_to']}<br>Capacity: {row['Flow capacity (kW)']} kW"
    ).add_to(map_fig)

# Add node markers
for idx, row in df_capacity_coords.iterrows():
    node_name = row['nodes']
    
    # Determine node type and styling
    if node_name.startswith('SH'):
        color = '#2ecc71'  # Green for Supply
        radius = 1
        node_type = 'Supply heat'
    elif node_name.startswith('SE'):
        color = "#2e38cc"  # Green for Supply
        radius = 1
        node_type = 'Supply electricity'
    elif node_name.startswith('D'):
        color = '#e74c3c'  # Red for Demand
        radius = 1
        node_type = 'Demand'
    else:  # TH or TE
        color = '#f39c12'  # Orange for Transmission
        radius = 1
        node_type = 'Transmission'
    
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=radius,
        popup=f"<b>{row['nodes']}</b> ({node_type})<br>Capacity: {row['Flow capacity (kW)']} kW",
        color=color,
        fill=True,
        fillColor=color,
        fillOpacity=0.8,
        weight=2
    ).add_to(map_fig)

# Display the map
map_fig.save("outputs/map_output.html")

In [17]:
# Export capacity data to csv
heat_export=df_heat.transpose()
heat_export.to_csv('outputs/heat_capacity.csv', header=['capacity_kw'])

electricity_export=df_electricity.transpose()
electricity_export.to_csv('outputs/electricity_capacity.csv', header=['capacity_kw'])